# Notebook 14 — Fannie Mae data compatibility and target validation
## Stage 1B — Schema alignment
The earlier inspection removed the first and last empty tokens. The official layout has a Reference Pool ID at position 1, not applicable to Single-Family Loan Performance. Removing the leading empty token may therefore have shifted the mapping. Keep every token and test this hypothesis before reading full loan histories.

Reference: [Fannie Mae glossary, July 29, 2024](https://capitalmarkets.fanniemae.com/sites/capmrkt/files/2024-07/crt-file-layout-and-glossary.pdf). [2026 disclosure changes](https://capitalmarkets.fanniemae.com/credit-risk-transfer/single-family-credit-risk-transfer/enhance-credit-risk-transfer-disclosures). The local download vintage is not yet known. This audit checks key positions and profiles every physical token, but does not certify the full release-specific schema or construct targets.

## 14.1 Locate the existing file
Use the same source path as the earlier inspection. No raw files will be changed.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from itertools import islice
import csv, json, re, zipfile

ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
SOURCE = ROOT / 'data/raw/Fannie_Mae/2018Q1/2018Q1.csv'
assert SOURCE.is_file(), f'File missing: {SOURCE}'
SIZE = SOURCE.stat().st_size
print('File:', SOURCE)
print('Bytes:', SIZE)

File: e:\capstone_project\mortgage-model-reliability\data\raw\Fannie_Mae\2018Q1\2018Q1.csv
Bytes: 6928164686


## 14.2 Inspect the physical positions
Read up to 200 records at each of seven offsets. Preserve leading and trailing empty fields. This is a diagnostic sample of monthly rows, not a random loan sample.

In [2]:
rows = []
widths = Counter()
for fraction in (0, .1, .3, .5, .7, .9, .99):
    with SOURCE.open('rb') as f:
        f.seek(int(SIZE * fraction))
        if fraction:
            f.readline()
        count = 0
        for raw in islice(f, 200):
            text = raw.decode('utf-8-sig').rstrip('\r\n')
            row = next(csv.reader([text], delimiter='|'))
            rows.append(row)
            widths[len(row)] += 1
            count += 1
        print(f'Offset {fraction:.0%}: {count} rows')
assert rows, 'No records found'
print('Physical token counts:', dict(widths))
print('Leading blank:', sum(r[0] == '' for r in rows))
print('Trailing blank:', sum(r[-1] == '' for r in rows))
assert len(widths) == 1, 'Inconsistent record widths; stop and inspect.'
WIDTH = len(rows[0])

Offset 0%: 200 rows
Offset 10%: 200 rows
Offset 30%: 200 rows
Offset 50%: 200 rows
Offset 70%: 200 rows
Offset 90%: 200 rows
Offset 99%: 200 rows
Physical token counts: {113: 1400}
Leading blank: 1400
Trailing blank: 1400


## 14.3 Define documented anchor positions
Use field positions from the 2024 glossary, not indices from the earlier stripped rows. The date, range and code tests below are diagnostic checks; they are not cleaning rules or final eligibility rules. Blank optional values are tracked separately.

In [3]:
def month(v):
    return bool(re.fullmatch(r'(0[1-9]|1[0-2])(19|20)\d{2}', v))

def number(v, low, high):
    try:
        return low <= float(v) <= high
    except ValueError:
        return False

# position: (field, test, required anchor)
FIELDS = {
    2: ('loan_id', lambda v: bool(re.fullmatch(r'[A-Za-z0-9]{1,12}', v)) and not month(v), True),
    3: ('report_month', month, True),
    4: ('channel', lambda v: v in {'R','C','B'}, True),
    8: ('original_rate', lambda v: number(v, .01, 30), True),
    10: ('original_upb', lambda v: number(v, 1, 100000000), True),
    13: ('original_term', lambda v: v.isdigit() and number(v, 1, 600), True),
    14: ('origination_month', month, True),
    15: ('first_payment_month', month, True),
    16: ('loan_age', lambda v: bool(re.fullmatch(r'-?\d{1,3}', v)), False),
    19: ('maturity_month', month, False),
    20: ('ltv', lambda v: number(v, 1, 300), False),
    21: ('cltv', lambda v: number(v, 1, 300), False),
    22: ('borrowers', lambda v: v.isdigit() and number(v, 1, 99), True),
    23: ('dti', lambda v: number(v, 0, 99), False),
    24: ('borrower_fico', lambda v: v.isdigit() and number(v, 300, 850), False),
    25: ('coborrower_fico', lambda v: v.isdigit() and number(v, 300, 850), False),
    26: ('first_time_homebuyer', lambda v: v in {'Y','N'}, False),
    27: ('purpose', lambda v: v in {'C','R','P','U'}, True),
    28: ('property_type', lambda v: v in {'CO','CP','PU','MH','SF'}, True),
    29: ('units', lambda v: v in {'1','2','3','4'}, True),
    30: ('occupancy', lambda v: v in {'P','S','I','U'}, True),
    31: ('state', lambda v: bool(re.fullmatch(r'[A-Z]{2}', v)), True),
    32: ('msa', lambda v: bool(re.fullmatch(r'\d{5}', v)), False),
    33: ('zip_short', lambda v: bool(re.fullmatch(r'\d{3}', v)), False),
    34: ('mi_percent', lambda v: number(v, 0, 100), False),
    35: ('amortization_type', lambda v: v in {'FRM','ARM'}, True),
    37: ('interest_only', lambda v: v in {'Y','N'}, False),
    40: ('delinquency', lambda v: v == 'XX' or bool(re.fullmatch(r'\d{1,2}', v)), False),
    41: ('payment_history', lambda v: len(v) <= 48 and len(v) % 2 == 0 and bool(re.fullmatch(r'(?:\d{2}|XX)+', v)), False),
    42: ('modification', lambda v: v in {'Y','N'}, False),
    44: ('zero_balance_code', lambda v: v.isdigit() and len(v) <= 3, False),
    45: ('zero_balance_month', month, False),
    51: ('last_paid_installment', month, False),
    52: ('foreclosure_month', month, False),
    53: ('disposition_month', month, False),
}
print('Anchor fields:', len(FIELDS))

Anchor fields: 35


## 14.4 Compare indexing hypotheses
Offset 0 means glossary position p is Python index p-1. Offset +1 reproduces the earlier leading-token removal. Offset -1 tests the alternative. A blank optional field is not evidence of correct alignment. Do not automatically select a layout based on scores.

In [4]:
audit = []
for offset in (-1, 0, 1):
    for pos, (name, check, required) in FIELDS.items():
        idx = pos - 1 + offset
        assert 0 <= idx < WIDTH
        vals = [r[idx].strip() for r in rows]
        present = [v for v in vals if v]
        valid = sum(check(v) for v in present)
        audit.append(dict(offset=offset, glossary_position=pos, physical_position=idx+1,
                          field=name, required_anchor=required, rows=len(vals),
                          blank=len(vals)-len(present), nonblank=len(present),
                          valid_nonblank=valid, invalid_nonblank=len(present)-valid,
                          valid_fraction_nonblank=round(valid/len(present), 5) if present else None))
summary = []
for offset in (-1, 0, 1):
    anchors = [a for a in audit if a['offset']==offset and a['required_anchor']]
    passed = sum(a['valid_nonblank'] / a['rows'] >= .99 for a in anchors)
    summary.append(dict(offset=offset, anchors_passing=passed, anchors_total=len(anchors)))
    print(summary[-1])
print('Offset 0 detail:')
for a in audit:
    if a['offset']==0:
        print(a['field'], 'present:', a['nonblank'], 'invalid:', a['invalid_nonblank'])

{'offset': -1, 'anchors_passing': 1, 'anchors_total': 15}
{'offset': 0, 'anchors_passing': 15, 'anchors_total': 15}
{'offset': 1, 'anchors_passing': 2, 'anchors_total': 15}
Offset 0 detail:
loan_id present: 1400 invalid: 0
report_month present: 1400 invalid: 0
channel present: 1400 invalid: 0
original_rate present: 1400 invalid: 0
original_upb present: 1400 invalid: 0
original_term present: 1400 invalid: 0
origination_month present: 1400 invalid: 0
first_payment_month present: 1400 invalid: 0
loan_age present: 1376 invalid: 0
maturity_month present: 1376 invalid: 0
ltv present: 1400 invalid: 0
cltv present: 1400 invalid: 0
borrowers present: 1400 invalid: 0
dti present: 1400 invalid: 0
borrower_fico present: 1395 invalid: 0
coborrower_fico present: 515 invalid: 0
first_time_homebuyer present: 1400 invalid: 0
purpose present: 1400 invalid: 0
property_type present: 1400 invalid: 0
units present: 1400 invalid: 0
occupancy present: 1400 invalid: 0
state present: 1400 invalid: 0
msa present

## 14.5 Profile every physical field
Export only aggregate token characteristics, not loan identifiers, individual rows or raw values. Positions beyond the documented anchors remain unverified, including trailing empty fields and any release-specific additions.

In [5]:
profile = []
for j in range(WIDTH):
    vals = [r[j].strip() for r in rows]
    nonblank = [v for v in vals if v]
    profile.append(dict(physical_position=j+1,
                        candidate_name=FIELDS.get(j+1, ('unverified',))[0],
                        rows=len(vals), blank=len(vals)-len(nonblank),
                        numeric_tokens=sum(bool(re.fullmatch(r'-?\d+(?:\.\d+)?', v)) for v in nonblank),
                        month_tokens=sum(month(v) for v in nonblank),
                        min_length=min(map(len, nonblank), default=0),
                        max_length=max(map(len, nonblank), default=0)))
print('Profiled physical positions:', len(profile))
print('Fully blank sampled positions:', [p['physical_position'] for p in profile if p['blank']==len(rows)])

Profiled physical positions: 113
Fully blank sampled positions: [1, 7, 11, 38, 39, 43, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 75, 76, 77, 78, 82, 83, 84, 85, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 104, 107, 108, 110, 112, 113]


## 14.6 Save the review evidence
The output is a small aggregate ZIP. Review the alignment and release vintage before approving a schema. Do not infer terminal event equivalence or build the target in this stage.

In [6]:
OUT = ROOT / 'data/processed/Fannie_Mae/notebook_14_compatibility' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT.mkdir(parents=True, exist_ok=False)
for name, records in [('anchor_checks.csv', audit), ('offset_comparison.csv', summary), ('physical_position_profile.csv', profile)]:
    with (OUT/name).open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
metadata = dict(source_name=SOURCE.name, file_bytes=SIZE, sampled_rows=len(rows),
                physical_width=WIDTH, width_counts=dict(widths),
                leading_blank=sum(r[0]=='' for r in rows), trailing_blank=sum(r[-1]=='' for r in rows),
                glossary_version='2024-07-29', local_release_vintage='unverified',
                status='pending_manual_review', target_constructed=False, models_scored=False)
(OUT/'audit_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
ZIP = OUT/'notebook_14_stage1b_review.zip'
with zipfile.ZipFile(ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p != ZIP:
            z.write(p, p.name)
print('Review ZIP:', ZIP)

Review ZIP: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_14_compatibility\20261124T143938077461Z\notebook_14_stage1b_review.zip


## Review gate
Share the ZIP. If the layout is ambiguous or release-specific fields are unresolved, obtain the glossary supplied with the local download before proceeding. No Freddie artifacts or Git files were modified by this audit.

## Stage 2 — Loan history and timing audit
Stage 1B review confirmed offset 0 (15/15 required anchors, 1,400 records, 113 physical tokens). This supports the positions used below, not the complete release-specific schema. We scan the entire 2018Q1 file once and retain all available rows for a deterministic approximately 1% sample of loan IDs, including noncontiguous records. The file quarter is an acquisition-file label, not an origination filter. This is a diagnostic pilot, not final cohort construction. No target labels or model scores are produced.

### 14.7 Configure the audit
Confirm the source matches the reviewed file and preserve the original physical positions. The 24-month windows below are diagnostic alternatives, not approved target definitions.

In [7]:
from collections import defaultdict, Counter
from functools import lru_cache
import hashlib
import pandas as pd

assert SIZE == 6928164686 and WIDTH == 113, 'Source differs from reviewed file; repeat schema review first.'
assert summary[1]['offset'] == 0 and summary[1]['anchors_passing'] == summary[1]['anchors_total'] == 15
POSITIONS = {'loan_id':2, 'report':3, 'orig':14, 'first_payment':15,
             'age':16, 'delinq':40, 'payment_history':41, 'modification':42,
             'zero_code':44, 'zero_month':45}
SAMPLE_MODULUS = 100  # approximately 1% of distinct loan IDs
MAX_SAMPLE_ROWS = 1_000_000  # stop rather than silently truncate histories
SOURCE_SIGNATURE = (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns)

@lru_cache(maxsize=100_000)
def selected(loan_id):
    return int.from_bytes(hashlib.blake2b(loan_id, digest_size=8, person=b'Fannie14v1').digest(), 'big') % SAMPLE_MODULUS == 0

print('Sample: deterministic loan-ID hash, approximately 1%')
print('Audit positions:', POSITIONS)

Sample: deterministic loan-ID hash, approximately 1%
Audit positions: {'loan_id': 2, 'report': 3, 'orig': 14, 'first_payment': 15, 'age': 16, 'delinq': 40, 'payment_history': 41, 'modification': 42, 'zero_code': 44, 'zero_month': 45}


### 14.8 Scan the source and collect complete sampled histories
Read every physical row. Keep only the selected columns for sampled IDs. Count malformed rows separately; stop if any prevent a reliable audit. The sample contains all available records for selected loans in this file, not necessarily their entire lifetime.

In [8]:
def scan_source(path, expected_width, progress_every=2_000_000):
    counts = Counter()
    sampled = []
    widths_seen = Counter()
    id_pattern = re.compile(rb'[A-Za-z0-9]{1,12}')
    indexes = [p-1 for p in POSITIONS.values()]
    with path.open('rb') as stream:
        for line_number, raw in enumerate(stream, 1):
            parts = raw.rstrip(b'\r\n').split(b'|')
            if line_number == 1:
                parts[0] = parts[0].removeprefix(b'\xef\xbb\xbf')
            counts['physical_rows'] += 1
            widths_seen[len(parts)] += 1
            if len(parts) != expected_width:
                counts['wrong_width'] += 1
                continue
            loan_id = parts[1].strip()
            if not id_pattern.fullmatch(loan_id):
                counts['invalid_loan_id'] += 1
                continue
            counts['valid_id_rows'] += 1
            if selected(loan_id):
                sampled.append(tuple(parts[i].strip().decode('ascii') for i in indexes))
                if len(sampled) > MAX_SAMPLE_ROWS:
                    raise RuntimeError('Sample exceeds memory guard; increase SAMPLE_MODULUS and rerun the entire scan.')
            if progress_every and line_number % progress_every == 0:
                print(f'{line_number:,} rows scanned; {len(sampled):,} sampled rows')
    return pd.DataFrame(sampled, columns=list(POSITIONS)), counts, widths_seen

history, scan_counts, full_widths = scan_source(SOURCE, WIDTH)
assert SOURCE_SIGNATURE == (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns), 'Source changed during scan.'
print(dict(scan_counts), dict(full_widths))
assert not scan_counts['wrong_width'] and not scan_counts['invalid_loan_id'], 'Malformed records found; review before continuing.'
assert not history.empty, 'No sampled rows'
print('Sample rows:', len(history), '| Sample distinct loans:', history.loan_id.nunique())

2,000,000 rows scanned; 22,660 sampled rows
4,000,000 rows scanned; 43,404 sampled rows
6,000,000 rows scanned; 63,865 sampled rows
8,000,000 rows scanned; 83,683 sampled rows
10,000,000 rows scanned; 103,966 sampled rows
12,000,000 rows scanned; 123,590 sampled rows
14,000,000 rows scanned; 145,624 sampled rows
16,000,000 rows scanned; 165,802 sampled rows
18,000,000 rows scanned; 186,285 sampled rows
20,000,000 rows scanned; 206,376 sampled rows
22,000,000 rows scanned; 229,575 sampled rows
{'physical_rows': 23563399, 'valid_id_rows': 23563399} {113: 23563399}
Sample rows: 245552 | Sample distinct loans: 4749


### 14.9 Check date fields, duplicates and reporting codes
Preserve blank and special delinquency values as separate categories. A numeric status of 3 or more is only a candidate serious-delinquency observation here; no event classification or terminal-code mapping is approved in this stage.

In [9]:
def month_index(series):
    valid = series.str.fullmatch(r'(0[1-9]|1[0-2])(19|20)\d{2}')
    safe = series.where(valid)
    return (pd.to_numeric(safe.str[2:]) * 12 + pd.to_numeric(safe.str[:2]) - 1).astype('Int64')

def month_text(i):
    if pd.isna(i): return 'missing'
    return f'{int(i)//12:04d}-{int(i)%12+1:02d}'

for name in ['report','orig','first_payment','zero_month']:
    history[name+'_idx'] = month_index(history[name])
    print(name, 'blank:', int(history[name].eq('').sum()), 'invalid nonblank:', int((history[name].ne('') & history[name+'_idx'].isna()).sum()))

history['numeric_delinq'] = pd.to_numeric(history.delinq.where(history.delinq.str.fullmatch(r'\d{1,2}')), errors='coerce')
valid_report = history[history.report_idx.notna()]
# Count duplicates without discarding conflicting rows.
duplicate_counts = valid_report.groupby(['loan_id','report_idx']).size()
duplicate_groups = duplicate_counts[duplicate_counts > 1]
print('Duplicate loan-month groups:', len(duplicate_groups), '| Extra records:', int((duplicate_groups-1).sum()))
print('Exact repeated selected-field records:', int(history[list(POSITIONS)].duplicated().sum()))
print('Delinquency codes (row counts):')
print(history.delinq.replace('', '<blank>').value_counts().to_string())
print('Zero balance codes (row counts):')
print(history.zero_code.replace('', '<blank>').value_counts().to_string())

report blank: 0 invalid nonblank: 0
orig blank: 0 invalid nonblank: 0
first_payment blank: 0 invalid nonblank: 0
zero_month blank: 242005 invalid nonblank: 0
Duplicate loan-month groups: 0 | Extra records: 0
Exact repeated selected-field records: 0
Delinquency codes (row counts):
delinq
00    237582
01      2676
XX       974
02       939
03       528
04       411
05       354
06       304
07       222
08       200
09       187
10       152
11       130
12       128
13       106
14        95
15        73
16        64
17        58
18        50
19        47
20        37
21        29
23        28
22        19
24        18
25        13
29        12
26        11
27         9
28         9
30         9
31         8
32         8
33         8
35         8
34         7
36         6
37         5
38         5
39         4
40         4
41         3
42         3
43         2
44         1
45         1
46         1
47         1
48         1
49         1
50         1
Zero balance codes (row counts):
zer

### 14.10 Audit timing at loan level
Check dates for consistency across a loan's records. Count missing months within the observed span and in each candidate 24-month window: origination + 1 through +24, and first payment through +23. These are calendar-month coverage diagnostics only; early termination, special statuses and left truncation need separate review.

In [10]:
def single_date(group, col):
    vals = group[col].dropna().unique()
    return int(vals[0]) if len(vals)==1 else None

def window_stats(months, start, g):
    if start is None:
        return [None]*6
    wanted = set(range(start, start+24))
    observed = months & wanted
    w = g[g.report_idx.between(start, start+23).fillna(False)]
    known = set(w.loc[w.numeric_delinq.notna(), 'report_idx'].astype(int))
    # Not a target: just counts of candidate observations.
    candidate = bool(w.numeric_delinq.ge(3).any())
    zero = g.loc[g.zero_code.ne(''), 'zero_month_idx'].dropna().astype(int)
    return [len(observed), len(known), int(candidate), int(any(start <= x <= start+23 for x in zero)),
            int(bool(months) and min(months)>start), int(bool(months) and max(months)<start+23)]

window_fields = ['observed_months','numeric_status_months','candidate_3plus','coded_zero_date_in_window','starts_late','ends_early']
loan_rows = []
for loan_id, g in history.groupby('loan_id', sort=False):
    months = set(g.report_idx.dropna().astype(int))
    orig = single_date(g,'orig_idx')
    fp = single_date(g,'first_payment_idx')
    first = min(months) if months else None
    last = max(months) if months else None
    z = g[g.zero_code.ne('')]
    zero_dates = set(z.zero_month_idx.dropna().astype(int))
    record = dict(orig_month=month_text(orig), first_payment_month=month_text(fp),
                  orig_date_conflict=int(g.orig_idx.nunique()>1), fp_date_conflict=int(g.first_payment_idx.nunique()>1),
                  orig_invalid_or_blank=int(g.orig_idx.isna().any()), fp_invalid_or_blank=int(g.first_payment_idx.isna().any()),
                  report_invalid_or_blank=int(g.report_idx.isna().any()),
                  rows=len(g), distinct_months=len(months),
                  duplicate_extra=int(g.report_idx.notna().sum()-len(months)),
                  internal_gap_months=(last-first+1-len(months)) if months else None,
                  first_report_minus_orig=first-orig if first is not None and orig is not None else None,
                  first_report_minus_fp=first-fp if first is not None and fp is not None else None,
                  fp_minus_orig=fp-orig if fp is not None and orig is not None else None,
                  any_zero_code=int(not z.empty), multiple_zero_codes=int(z.zero_code.nunique()>1),
                  multiple_zero_dates=int(len(zero_dates)>1),
                  zero_code_without_valid_date=int(z.zero_month_idx.isna().any()),
                  zero_date_without_code=int((g.zero_code.eq('') & g.zero_month_idx.notna()).any()),
                  report_after_zero_date=int(bool(months and zero_dates) and max(months)>min(zero_dates)))
    for prefix, start in [('orig',None if orig is None else orig+1),('fp',fp)]:
        record.update({prefix+'_'+k:v for k,v in zip(window_fields,window_stats(months,start,g))})
    loan_rows.append(record)
loans = pd.DataFrame(loan_rows)
assert len(loans)==history.loan_id.nunique()
print('Loans summarized:', len(loans))
print(loans[['rows','distinct_months','internal_gap_months','first_report_minus_orig','first_report_minus_fp','fp_minus_orig']].describe().round(2).to_string())

Loans summarized: 4749
          rows  distinct_months  internal_gap_months  first_report_minus_orig  first_report_minus_fp  fp_minus_orig
count  4749.00          4749.00               4749.0                  4745.00                4748.00        4744.00
mean     51.71            51.71                  0.0                     1.19                  -0.81           2.01
std      31.23            31.23                  0.0                     0.85                   0.85           0.19
min       2.00             2.00                  0.0                     0.00                  -2.00           1.00
25%      28.00            28.00                  0.0                     1.00                  -1.00           2.00
50%      39.00            39.00                  0.0                     1.00                  -1.00           2.00
75%      97.00            97.00                  0.0                     2.00                   0.00           2.00
max      99.00            99.00                  

### 14.11 Summarize the audit
Report loan-level and record-level counts separately. Origination quarters are derived from actual origination dates, not the filename. The presence of a zero-balance code does not establish a credit event.

In [11]:
def distribution(series, field):
    counts = series.astype('string').fillna('<missing>').value_counts(dropna=False)
    return pd.DataFrame({'field':field,'value':counts.index.astype(str),'count':counts.values})

orig_quarter = pd.to_datetime(loans.orig_month, format='%Y-%m', errors='coerce').dt.to_period('Q').astype('string')
origination_counts = distribution(orig_quarter,'origination_quarter')
timing_columns = ['first_report_minus_orig','first_report_minus_fp','fp_minus_orig','internal_gap_months','distinct_months']
timing_dist = pd.concat([distribution(loans[c],c) for c in timing_columns], ignore_index=True)
flags = ['orig_date_conflict','fp_date_conflict','orig_invalid_or_blank','fp_invalid_or_blank','report_invalid_or_blank','any_zero_code','multiple_zero_codes','multiple_zero_dates','zero_code_without_valid_date','zero_date_without_code','report_after_zero_date']
loan_checks = pd.DataFrame([{'check':c,'loans':int(loans[c].sum()),'sample_loans':len(loans)} for c in flags]+[
    {'check':'any_duplicate_month','loans':int(loans.duplicate_extra.gt(0).sum()),'sample_loans':len(loans)},
    {'check':'any_internal_gap','loans':int(loans.internal_gap_months.gt(0).sum()),'sample_loans':len(loans)}])
window_summary = []
for prefix in ['orig','fp']:
    c = loans[prefix+'_observed_months']
    valid = c.notna()
    window_summary.append(dict(window=prefix, sample_loans=len(loans), date_available=int(valid.sum()),
        all_24_reported=int(c.eq(24).sum()), all_24_numeric_status=int(loans[prefix+'_numeric_status_months'].eq(24).sum()),
        fewer_than_24_reported=int((valid & c.lt(24)).sum()), candidate_3plus_loans=int(loans[prefix+'_candidate_3plus'].sum()),
        coded_zero_date_in_window=int(loans[prefix+'_coded_zero_date_in_window'].sum()),
        starts_late=int(loans[prefix+'_starts_late'].sum()), ends_early=int(loans[prefix+'_ends_early'].sum())))
window_summary = pd.DataFrame(window_summary)
window_distribution = pd.concat([distribution(loans[p+'_'+c],p+'_'+c) for p in ['orig','fp'] for c in ['observed_months','numeric_status_months']],ignore_index=True)
code_counts = pd.concat([distribution(history[c].replace('', '<blank>'),c) for c in ['delinq','zero_code']],ignore_index=True)
# Distinct loan counts per code; a loan may contribute to more than one code.
loan_code_counts = pd.concat([distribution(history[['loan_id',c]].drop_duplicates()[c].replace('', '<blank>'),c) for c in ['delinq','zero_code']],ignore_index=True)
print(origination_counts.to_string(index=False))
print(loan_checks.to_string(index=False))
print(window_summary.to_string(index=False))

              field     value  count
origination_quarter    2018Q1   2656
origination_quarter    2017Q4   2068
origination_quarter    2017Q3     18
origination_quarter <missing>      4
origination_quarter    2017Q2      2
origination_quarter    2017Q1      1
                       check  loans  sample_loans
          orig_date_conflict      4          4749
            fp_date_conflict      1          4749
       orig_invalid_or_blank      0          4749
         fp_invalid_or_blank      0          4749
     report_invalid_or_blank      0          4749
               any_zero_code   3547          4749
         multiple_zero_codes      0          4749
         multiple_zero_dates      0          4749
zero_code_without_valid_date      0          4749
      zero_date_without_code      0          4749
      report_after_zero_date      0          4749
         any_duplicate_month      0          4749
            any_internal_gap      0          4749
window  sample_loans  date_available  all

### 14.12 Export aggregate evidence
Save aggregate summaries only. No loan identifiers, row-level data, targets or predictions are exported.

In [12]:
OUT2 = ROOT / 'data/processed/Fannie_Mae/notebook_14_compatibility' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT2.mkdir(parents=True, exist_ok=False)
exports = {'origination_quarter_counts.csv':origination_counts, 'timing_distributions.csv':timing_dist,
           'loan_audit_checks.csv':loan_checks,'window_coverage_summary.csv':window_summary,
           'window_coverage_distributions.csv':window_distribution,'record_code_counts.csv':code_counts,
           'loan_code_counts.csv':loan_code_counts}
for filename, table in exports.items():
    table.to_csv(OUT2/filename,index=False)
meta2 = dict(stage='2', source_name=SOURCE.name, file_bytes=SIZE, physical_width=WIDTH,
             full_scan_counts=dict(scan_counts), full_width_counts=dict(full_widths), sample_modulus=SAMPLE_MODULUS,
             sample_method='BLAKE2b loan ID modulo; all matching rows in file', sample_rows=len(history), sample_loans=len(loans),
             duplicate_loan_month_groups=len(duplicate_groups), duplicate_extra_rows=int((duplicate_groups-1).sum()),
             report_month_min=month_text(history.report_idx.min()), report_month_max=month_text(history.report_idx.max()),
             invalid_or_blank_dates={c:int(history[c+'_idx'].isna().sum()) for c in ['report','orig','first_payment','zero_month']},
             invalid_nonblank_dates={c:int((history[c].ne('') & history[c+'_idx'].isna()).sum()) for c in ['report','orig','first_payment','zero_month']},
             local_release_vintage='unverified', target_constructed=False, models_scored=False,
             window_definitions={'orig':'origination +1 through +24','fp':'first payment +0 through +23'}, status='pending_manual_review')
(OUT2/'stage2_metadata.json').write_text(json.dumps(meta2,indent=2),encoding='utf-8')
ZIP2=OUT2/'notebook_14_stage2_review.zip'
with zipfile.ZipFile(ZIP2,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT2.iterdir()):
        if f != ZIP2: z.write(f,f.name)
print('Review ZIP:', ZIP2)

Review ZIP: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_14_compatibility\20261124T144309949497Z\notebook_14_stage2_review.zip


## Stage 2 review gate
Share `notebook_14_stage2_review.zip`. Review timing, duplicate months, special status values and terminal codes against the appropriate documentation before agreeing on a target. A complete reporting window is not the same as a valid outcome label, and early termination is not automatically ineligible. Do not push or score models yet.